# Kuroshio R5.6 — 双分支几何表示的固定模型对照资格实验

**目的：** 检验 R5.5 发现的旧表示偏离流轴交点问题，是否能通过沿流轴弧长单调的经线交点单值化（DP）在**不换训练协议、不增减样本、不篡改冻结数据**的条件下带来可测的预测收益。

- 实验两套表示：**Original 原双分支冻结表示** / **DP-B2 修复第二分支（第一分支完全相同）**。
- 固定网络：原同规模 LSTM、与 PCConcat 参数匹配的 LSTM、原 EOF-TCN-LSTM、R53-ChannelConv-PCConcat。每个网络每个表示用完全相同的开发 QA 年份、种子、Epoch 和早停协议。
- 评价三种口径：A **同一原冻结参考**（可比较）、B **各自编码参考**（仅诊断）、C **逐日真实跟踪流轴参考**（抽样，属于流水线生成的内部参考，不是独立观测）。
- **重要：** 这是机制验证，不是正式多年份盲测。2008/2016/2012/2020 已被开发过程接触。
- 训练前先对全部 11,771 日生成 DP 候选并审计时间连续性；候选不合格则仅归档，不长训。不能利用模型误差选择 DP 分支。
- 失败、异常与无提升都自动写入 ZIP + SHA256；Jupyter 显示本机 FileLink，VS Code 如果不能打开请直接上传 D 盘 ZIP。

In [ ]:
# ===== 0. 配置：先核实三个目录；改好后从头顺序运行 =====
from dataclasses import dataclass, asdict
from pathlib import Path
import os, json, math, gc, time, random, hashlib, warnings, sys
import numpy as np
import pandas as pd

@dataclass
class CFG:
    ROOT: str = '/mnt/d/KuroshioDaily1993_2025'
    DATASET_REL: str = 'final/dataset/daily_dual_branch_paths_1993_2025_FINAL.npz'
    HANDOFF_REL: str = 'final/reports/LONG_DAILY_1993_2025_HANDOFF.json'
    QC_INDEX_REL: str = 'final/reports/最终逐日路径索引.csv'
    MANUAL_QUEUE_REL: str = 'reports/07_人工复核队列.csv'
    MANUAL_METADATA_REL: str = 'manual_qc/metadata'
    OUTPUT_REL: str = 'R5_6_路径表示固定模型对照资格实验'
    HISTORY: int = 180
    LEADS: tuple = (30, 60, 90)
    TEST_YEARS: tuple = (2005, 2009, 2013, 2017, 2021, 2024)
    PURGE_DAYS: int = 7
    MAX_INPUT_GAP: int = 2
    MAX_MISSING_FRAC: float = 0.03
    MIN_DATA_YEARS: int = 25
    MIN_ANNUAL_COVERAGE: float = 0.85
    MIN_TRAIN: int = 500
    MIN_VAL: int = 100
    MIN_TEST: int = 100
    DROP_BRANCH1_FIRST: int = 12  # 43->31; same old representation
    BRANCH2_MAX_LON: float = 145.0
    EOF_VARIANCE_TARGET: float = 0.99
    BATCH_SIZE: int = 32
    EPOCHS_B1: int = 70
    EPOCHS_B2: int = 70
    PATIENCE: int = 12
    LR: float = 0.001
    WEIGHT_DECAY: float = 0.0001
    DROPOUT: float = 0.30
    SEEDS: tuple = (20,21,22,23,24)
    AR_LAGS: tuple = (3, 7)
    AR_ALPHAS: tuple = (0.1, 1., 10.)
    CLIM_HALF_WINDOW_DAYS: int = 15
    CLIM_MIN_SAMPLES: int = 10
    BOOT_N: int = 1000
    BOOT_BLOCK_DAYS: int = 30
    FORCE_TRAIN: bool = False
    USE_AMP: bool = True
    RUN_FULL_TRAINING: bool = False   # R5.3仅机制资格实验，严禁直接转长训
    TRAIN_OTHER_HYBRIDS: bool = False # 下一轮网络结构比较可启用，不改baseline主表
    FIG_DPI: int = 300
    PILOT_TEST_YEARS: tuple = (2009, 2017)  # QA: 2008/2016, inner early-stop: 2007/2015
    PILOT_SEEDS: tuple = (20, 21)
    PILOT_EPOCHS: int = 24
    PILOT_PATIENCE: int = 7
    GATE_MIN_GAIN_KM: float = 0.50
    GATE_MAX_LEAD_REGRESSION_KM: float = 2.0

    CONFIRM_PILOT_YEARS: tuple = (2013, 2021)  # QA 2012/2020; 曾经被研发流程涉及，不能称独立盲测
    MAX_PARAM_OVERHEAD_RATIO: float = 0.15
    REQUIRE_ALL_LEADS_NONINFERIOR: bool = True
    REQUIRE_BOTH_QA_YEARS_GAIN: bool = True
    ARCHIVE_INCLUDE_WEIGHTS: bool = False
    ARCHIVE_INCLUDE_CACHE: bool = False
    ARCHIVE_MAX_FILE_MB: int = 150
    RUN_CONFIRMATION: bool = False   # 保留配置但本轮不实现额外确认；改True也不会自动长训
    RAW_TRACK_DIR_REL: str = 'final/tracks_final'
    MAX_MISSING_DP_DAY_FRAC: float = 0.005
    MAX_DP_TEMPORAL_P95_RATIO: float = 1.40
    RAW_EVAL_ISSUES_PER_QA_YEAR: int = 36
    REBUILD_DP: bool = False
    DP_REPORT_EVERY: int = 250
    ARCHIVE_INCLUDE_WEIGHTS: bool = False
    ARCHIVE_INCLUDE_CACHE: bool = False

cfg=CFG(); ROOT=Path(cfg.ROOT); DS=ROOT/cfg.DATASET_REL; OUT=ROOT/cfg.OUTPUT_REL
DATA=OUT/'实验数据'; CKPT=OUT/'模型权重'; PRED=OUT/'逐样本预测'; REP=OUT/'报告'; FZH=OUT/'图表'/'中文'; FEN=OUT/'图表'/'英文'
for d in (DATA,CKPT,PRED,REP,FZH,FEN): d.mkdir(parents=True,exist_ok=True)
print(json.dumps(asdict(cfg),indent=2,ensure_ascii=False))
ACTIVE_REP='Original'
RAW_TRACK_DIR=ROOT/cfg.RAW_TRACK_DIR_REL
if not RAW_TRACK_DIR.is_dir(): raise FileNotFoundError(f'找不到原始完整流轴目录: {RAW_TRACK_DIR}')
DERIVED=OUT/'派生表示_不覆盖冻结数据';DERIVED.mkdir(exist_ok=True,parents=True)
ARCHIVE_PATH=ROOT/'R56_上传给ChatGPT_几何表示机制实验结果.zip'
ARCHIVE_SHA=Path(str(ARCHIVE_PATH)+'.sha256')
if not DS.exists(): raise FileNotFoundError(f'未发现真正的冻结数据：{DS}。请先运行人工QC notebook最后的冻结与数据构建单元。')

In [ ]:
# ===== 1. GPU与字体/依赖检查（不合格立即停止，避免白训练）=====
import importlib.util
required=('torch','scipy','sklearn','matplotlib','tqdm','pandas','numpy')
missing=[v for v in required if importlib.util.find_spec(v) is None]
if missing: raise RuntimeError('WSL环境缺依赖：'+', '.join(missing)+'；用当前Kernel对应python执行 python -m pip install torch scipy scikit-learn matplotlib tqdm pandas numpy')
import torch, torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from scipy.spatial import cKDTree
from sklearn.linear_model import Ridge
import matplotlib, matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.font_manager import FontProperties
from tqdm.auto import tqdm
print('kernel=',sys.executable,'torch=',torch.__version__,'CUDA=',torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('本轮RTX5080正式训练要求CUDA GPU；请检查WSL nvidia-smi 与 PyTorch CUDA构建。')
print('GPU=',torch.cuda.get_device_name(0)); DEV=torch.device('cuda')
torch.set_float32_matmul_precision('high')
font_paths=['/mnt/c/Windows/Fonts/msyh.ttc','/mnt/c/Windows/Fonts/simhei.ttf','/mnt/c/Windows/Fonts/simsun.ttc','/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc']
CJK_NAME=None
for path in font_paths:
    if Path(path).exists():
        try:
            font_manager.fontManager.addfont(path)
            CJK_NAME=FontProperties(fname=path).get_name(); break
        except Exception: pass
if CJK_NAME is None:
    raise RuntimeError('中文字体缺失：请在WSL安装 sudo apt install fonts-noto-cjk 或检查 Windows /mnt/c/Windows/Fonts，避免绘图乱码。')
print('Chinese font=',CJK_NAME)

In [ ]:
# ===== 2. 最终数据读取、年度覆盖、人工QC冻结核对 =====
with np.load(DS,allow_pickle=False) as z:
    dates=pd.DatetimeIndex(pd.to_datetime(z['dates'])).normalize()
    b1=np.asarray(z['branch1_lon'],np.float32)
    lat1=np.asarray(z['branch1_target_lats'],float)
    b2=np.asarray(z['branch2_lat'],np.float32)
    lon2=np.asarray(z['branch2_target_lons'],float)
    source=np.asarray(z['source']).astype(str) if 'source' in z.files else np.array(['unknown']*len(dates))
if not (len(dates)==len(b1)==len(b2)==len(source)): raise RuntimeError('dates / 两分支 / source 数量不一致')
if dates.has_duplicates: raise RuntimeError('发现重复日期，禁止将其直接用于训练')
ord=np.argsort(dates.values); dates=dates[ord]; b1=b1[ord];b2=b2[ord];source=source[ord]
if not np.isfinite(b1).all() or not np.isfinite(b2).all(): raise RuntimeError('冻结数据中含 NaN/Inf')

# 物理范围异常只作质控警示（不随意删除或修改路径），便于发现断面插值外推造成的不可信日
suspicious=(~((b1>=119)&(b1<=140)).all(axis=1)) | (~((b2>=15)&(b2<=50)).all(axis=1))
if suspicious.any():
    suspect_df=pd.DataFrame({'异常日期':dates[suspicious], '原因':'双分支坐标超出宽松物理范围，需人工核对'})
    suspect_df.to_csv(DATA/'01_待人工复查的物理范围异常路径.csv',index=False,encoding='utf-8-sig')
    warnings.warn(f'发现 {suspicious.sum()} 天路径越过宽松物理边界，建议正式训练前核实此文件')

if b1.shape[1]<=cfg.DROP_BRANCH1_FIRST: raise RuntimeError('分支1维数过小')
b1=b1[:,cfg.DROP_BRANCH1_FIRST:];lat1=lat1[cfg.DROP_BRANCH1_FIRST:]
keep2=lon2<=cfg.BRANCH2_MAX_LON+1e-8;b2=b2[:,keep2];lon2=lon2[keep2]
if len(b1)!=len(b2) or len(lat1)!=b1.shape[1] or len(lon2)!=b2.shape[1]: raise RuntimeError('双分支物理坐标不匹配')
if dates.min().year>1993 or dates.max().year<2024: raise RuntimeError('数据不覆盖1993–2024所需区间，不可宣称多年全历史验证')
cal=pd.date_range(dates.min(),dates.max(),freq='D'); pos=cal.get_indexer(dates)
valid=np.zeros(len(cal),bool);valid[pos]=True
A1=np.full((len(cal),b1.shape[1]),np.nan,np.float32);A2=np.full((len(cal),b2.shape[1]),np.nan,np.float32)
A1[pos]=b1; A2[pos]=b2
# causal forward fill (ONLY history, never target). All targets later read A1/A2 raw by exact date.
def causal_fill(raw,limit): return pd.DataFrame(raw).ffill(limit=limit).to_numpy(dtype=np.float32)
F1=causal_fill(A1,cfg.MAX_INPUT_GAP);F2=causal_fill(A2,cfg.MAX_INPUT_GAP)
annual=pd.DataFrame({'日期':cal,'有效':valid});annual['年份']=annual['日期'].dt.year
cover=annual.groupby('年份',as_index=False).agg(理论天数=('日期','size'),有效天数=('有效','sum'))
cover['有效比例']=cover['有效天数']/cover['理论天数'];cover.to_csv(DATA/'01_逐年覆盖率.csv',index=False,encoding='utf-8-sig')
if (cover['有效比例']>=cfg.MIN_ANNUAL_COVERAGE).sum()<cfg.MIN_DATA_YEARS:
    raise RuntimeError('具备≥85%有效覆盖的年份不足25年，请优先排查未完成人工修复或缺失年；详情见01_逐年覆盖率.csv')
hand=ROOT/cfg.HANDOFF_REL
if hand.exists():
    ho=json.loads(hand.read_text(encoding='utf-8'))
    print('HANDOFF final_daily_paths=',ho.get('final_daily_paths'))
    if int(ho.get('final_daily_paths',-1))!=len(dates): raise RuntimeError('HANDOFF与冻结npz样本数不一致')
else: warnings.warn('未发现HANDOFF.json；冻结数据可供资格检查，但交接记录尚未核实')
idxp=ROOT/cfg.QC_INDEX_REL
if idxp.exists():
    idx=pd.read_csv(idxp)
    if 'status' in idx.columns:
        unresolved=idx[~idx['status'].isin(['final','missing_raw','unusable'])]
        if len(unresolved): raise RuntimeError(f'最终逐日索引仍有 {len(unresolved)} 条未处理状态，停止训练')
queue=ROOT/cfg.MANUAL_QUEUE_REL; mdir=ROOT/cfg.MANUAL_METADATA_REL
if queue.exists():
    qq=pd.read_csv(queue)
    qdates=pd.to_datetime(qq['date']).dt.strftime('%Y%m%d') if len(qq) else []
    missing_meta=[q for q in qdates if not (mdir/f'{q}.json').exists()]
    if missing_meta: raise RuntimeError(f'{len(missing_meta)} 个manual复核队列日期尚无处理metadata: {missing_meta[:10]}')
    bad_meta=[]
    allowed={'manual_corrected','accepted_auto_after_manual_qc','unusable'}
    for q in qdates:
        try:
            state=json.loads((mdir/f'{q}.json').read_text(encoding='utf-8')).get('status')
            if state not in allowed:bad_meta.append((q,state))
        except Exception as e:bad_meta.append((q,repr(e)))
    if bad_meta:raise RuntimeError(f'{len(bad_meta)} 条人工QC仍不是已确认状态: {bad_meta[:10]}')
# CMEMS来源切换边界连续性检查：仅警告，不把真实大蛇行直接当成数据损坏
edge=pd.Timestamp('2025-01-01')
prev=np.flatnonzero(dates<edge)
post=np.flatnonzero(dates>=edge)
if len(prev) and len(post):
    i,j=int(prev[-1]),int(post[0])
    if (dates[j]-dates[i]).days<=5:
        d1=float(np.sqrt(np.mean((b1[j]-b1[i])**2)))
        d2=float(np.sqrt(np.mean((b2[j]-b2[i])**2)))
        pd.DataFrame([{'产品交界前':dates[i],'交界后':dates[j],
                       '分支1逐日经度RMSE_度':d1,'分支2逐日纬度RMSE_度':d2}]).to_csv(
            DATA/'01_2024到2025产品交界连续性.csv',index=False,encoding='utf-8-sig')
        if d1>1.0 or d2>1.5:
            warnings.warn('2024-2025 CMEMS来源交界发生较大路径跳变，应先检查跨产品物理一致性与流轴QC')
print('DATA PASS:',len(dates),'days;',dates.min().date(),'to',dates.max().date(),'; B1',b1.shape[1],'; B2',b2.shape[1])
display(cover)

In [ ]:
# ===== 3. 创建样本索引，不展开 [样本,180,特征] 巨型数组 =====
# target index = issue+lead; test targets require true valid day. input missing <=2 continuous days.
L=max(cfg.LEADS); H=cfg.HISTORY
missing=(~valid).astype(np.int32)
issue_indices=[];audit=[]
for i in range(H-1,len(cal)-L):
    if not valid[i]: continue
    win=missing[i-H+1:i+1]
    if win.mean()>cfg.MAX_MISSING_FRAC: continue
    if len(win) and np.max(np.diff(np.flatnonzero(np.r_[1,win==0,1]))-1)>cfg.MAX_INPUT_GAP: continue
    if any(not valid[i+d] for d in cfg.LEADS): continue
    if not (np.isfinite(F1[i-H+1:i+1]).all() and np.isfinite(F2[i-H+1:i+1]).all()): continue
    issue_indices.append(i)
ISS=np.asarray(issue_indices,np.int32)
if len(ISS)==0: raise RuntimeError('没有有效180→30/60/90 anchors，需检查缺日与冻结数据')
anchor=pd.DataFrame({'起报日期':cal[ISS]})
for l in cfg.LEADS: anchor[f'{l}天目标日期']=cal[ISS+l]
anchor.to_csv(DATA/'02_全部有效预测起报日期.csv',index=False,encoding='utf-8-sig')
print('anchor count=',len(ISS),' first=',cal[ISS[0]],'last=',cal[ISS[-1]])

In [ ]:
# ===== 4. 真正的 expanding-window 多年后报：test year = 2005/2009/2013/2017/2021/2024 =====
FOLDS=[];fr=[]; splitrows=[]
for fold_id,year in enumerate(cfg.TEST_YEARS,1):
    valstart=pd.Timestamp(year-1,1,1); teststart=pd.Timestamp(year,1,1)
    testend=pd.Timestamp(year,12,31)
    train_cut=valstart-pd.Timedelta(days=cfg.PURGE_DAYS)
    val_cut=teststart-pd.Timedelta(days=cfg.PURGE_DAYS)
    issues=cal[ISS]; latest=cal[ISS+L]
    tr=ISS[(latest<train_cut)]
    va=ISS[(issues>=valstart)&(issues<teststart)&(latest<val_cut)]
    te=ISS[(issues>=teststart)&(issues<=testend)]
    if len(tr)<cfg.MIN_TRAIN or len(va)<cfg.MIN_VAL or len(te)<cfg.MIN_TEST:
        raise RuntimeError(f'Fold {year} 样本不足: tr={len(tr)} val={len(va)} test={len(te)}；请查该年的日路径覆盖和连续缺日')
    if not(cal[tr+L].max()<cal[va].min()-pd.Timedelta(days=cfg.PURGE_DAYS) and cal[va+L].max()<cal[te].min()-pd.Timedelta(days=cfg.PURGE_DAYS)):
        raise RuntimeError(f'Fold {year} 发现严格时序泄漏')
    if (len(set(cal[tr+L])&set(cal[va+L])) or len(set(cal[tr+L])&set(cal[te+L])) or len(set(cal[va+L])&set(cal[te+L]))):
        raise RuntimeError(f'Fold {year} label date overlap')
    ff={'fold':fold_id,'year':year,'train':tr,'val':va,'test':te,'train_cut':train_cut,'val_cut':val_cut}
    FOLDS.append(ff)
    fr.append({'折次':fold_id,'测试年份':year,'训练样本':len(tr),'验证样本':len(va),'测试样本':len(te),
               '训练最晚标签日期':cal[tr+L].max(),'验证最早起报日期':cal[va].min(),
               '验证最晚标签日期':cal[va+L].max(),'测试最早起报日期':cal[te].min(),'测试最晚标签日期':cal[te+L].max(),'因果审计':'PASS'})
    for label,ix in [('训练',tr),('验证',va),('测试',te)]:
        splitrows.extend({'折次':fold_id,'测试年份':year,'集合':label,'起报日期':str(cal[i].date()),'最晚目标日期':str(cal[i+L].date())} for i in ix)
foldtab=pd.DataFrame(fr); foldtab.to_csv(DATA/'03_多年滚动后报折次及防泄漏审计.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(splitrows).to_csv(DATA/'03_逐样本时序划分.csv',index=False,encoding='utf-8-sig')
print('[STRICT MULTI-YEAR CAUSAL HINDCAST] ALL PASS'); display(foldtab)

In [ ]:
# ===== R5.5 冻结、可复用的几何交点选择规则 =====
def haversine_segment_km(lon,lat):
    x1=np.radians(np.asarray(lon[:-1],float)); x2=np.radians(np.asarray(lon[1:],float))
    y1=np.radians(np.asarray(lat[:-1],float)); y2=np.radians(np.asarray(lat[1:],float))
    a=np.sin((y2-y1)/2)**2 + np.cos(y1)*np.cos(y2)*np.sin((x2-x1)/2)**2
    return 6371.0*2*np.arctan2(np.sqrt(np.clip(a,0,1)),np.sqrt(np.maximum(0,1-a)))

def all_crossings(lon,lat,grids):
    """每条固定经线与有向原始流轴的交点列表 (latitude, alongpath_km)。
    半开线段消除大部分共享端点重复；经度恒定线段不进入分母。
    """
    lon=np.asarray(lon,dtype=float);lat=np.asarray(lat,dtype=float);grids=np.asarray(grids,float)
    mask=np.isfinite(lon)&np.isfinite(lat)
    lon=lon[mask];lat=lat[mask]
    if len(lon)<3:raise ValueError('有效轨迹点数不足3')
    arclen=np.r_[0.,np.cumsum(haversine_segment_km(lon,lat))]
    x0,x1=lon[:-1],lon[1:]; y0,y1=lat[:-1],lat[1:]
    good=np.isfinite(x0)&np.isfinite(x1)&np.isfinite(y0)&np.isfinite(y1)&(np.abs(x1-x0)>1e-10)
    res=[]
    for g in grids:
        idx=np.flatnonzero(good&(((x0<=g)&(x1>g))|((x1<=g)&(x0>g))))
        if idx.size==0:
            res.append(np.empty((0,2),float));continue
        t=(g-x0[idx])/(x1[idx]-x0[idx])
        yi=y0[idx]+t*(y1[idx]-y0[idx])
        si=arclen[idx]+t*(arclen[idx+1]-arclen[idx])
        order=np.argsort(si)
        res.append(np.column_stack([yi,si])[order])
    return res

def latitude_crossings(lon,lat,grids):
    """B1 固定纬线的经度交点及真实地理弧长（不交换经纬度计算距离）。"""
    lon=np.asarray(lon,float);lat=np.asarray(lat,float);grids=np.asarray(grids,float)
    good=np.isfinite(lon)&np.isfinite(lat);lon=lon[good];lat=lat[good]
    if len(lat)<3:raise ValueError('有效流轴点不足3')
    arclen=np.r_[0.,np.cumsum(haversine_segment_km(lon,lat))]
    a=lat[:-1];b=lat[1:]; x=lon[:-1];y=lon[1:]
    res=[]
    for g in grids:
        idx=np.flatnonzero((((a<=g)&(b>g))|((b<=g)&(a>g))) & (np.abs(b-a)>1e-10))
        if len(idx)==0:res.append(np.empty((0,2),float));continue
        t=(g-a[idx])/(b[idx]-a[idx])
        coords=x[idx]+t*(y[idx]-x[idx])
        ss=arclen[idx]+t*(arclen[idx+1]-arclen[idx])
        order=np.argsort(ss)
        res.append(np.column_stack([coords,ss])[order])
    return res

def branch1_scores(lon,lat,lats_grid,old_lon):
    xs=latitude_crossings(lon,lat,lats_grid)
    counts=np.array([len(a) for a in xs])
    scales=111.2*np.cos(np.radians(lats_grid))
    nearest=np.array([np.min(np.abs(a[:,0]-v))*scale if len(a) and np.isfinite(v) else np.nan
                      for a,v,scale in zip(xs,old_lon,scales)],float)
    spans=np.array([np.ptp(a[:,0])*scale if len(a)>1 else np.nan for a,scale in zip(xs,scales)],float)
    ok=np.isfinite(nearest)
    return {'第一分支无交点比例_复核':float(np.mean(counts==0)),
            '第一分支多交点比例_复核':float(np.mean(counts>1)),
            '第一分支多交点纬线经度跨度均值_km':float(np.nanmean(spans)) if np.isfinite(spans).any() else 0.0,
            '第一分支旧表示距真实交点平均_km':float(np.nanmean(nearest)) if ok.any() else np.nan,
            '第一分支旧表示距真实交点P95_km':float(np.nanpercentile(nearest,95)) if ok.any() else np.nan}

def continuous_arc_selector(intersections, grid):
    """对有交点的相邻经线块进行单调弧长 DP。
    只解决一个**单值分支**的选择，不可恢复未被选择的合法蛇行分支。
    不允许后一经线沿原始流轴向后移动（s 减小）。
    """
    n=len(grid); selected=np.full(n,np.nan); picked_s=np.full(n,np.nan)
    costs=[];prev=[]; restart=0
    # 缺口中断一段 DP；不强行外推
    for i in range(n):
        arr=intersections[i]
        if len(arr)==0:
            costs.append(None);prev.append(None);continue
        if i==0 or len(intersections[i-1])==0:
            costs.append(np.zeros(len(arr),float)+0.001*arr[:,1]);prev.append(np.full(len(arr),-1,int))
            continue
        prev_arr=intersections[i-1];p_cost=costs[i-1]
        cost=np.full(len(arr),np.inf);link=np.full(len(arr),-1,int)
        for j,(y,s) in enumerate(arr):
            ds=s-prev_arr[:,1]
            ok=ds>=-0.001
            if not ok.any():continue
            # 左端变化只影响平滑性；过大的弧长跃迁代价更高，但不是硬性剪枝
            dy=111.2*np.abs(y-prev_arr[:,0])
            transition=dy + 0.07*np.maximum(ds,0) + 0.01*np.maximum(ds-120,0)
            cand=p_cost+transition
            cand=np.where(ok,cand,np.inf)
            k=int(np.argmin(cand))
            if np.isfinite(cand[k]):cost[j]=cand[k];link[j]=k
        if not np.isfinite(cost).any():
            restart+=1
            cost=np.zeros(len(arr),float)+0.001*arr[:,1]
            link[:]=-1
        costs.append(cost);prev.append(link)
    # 回溯每个连续子段，并允许不可达时被动拆段
    j=n-1
    while j>=0:
        if not len(intersections[j]):j-=1;continue
        c=costs[j];k=int(np.argmin(c))
        while j>=0 and len(intersections[j]):
            if not np.isfinite(c[k]):break
            selected[j]=intersections[j][k,0];picked_s[j]=intersections[j][k,1]
            parent=int(prev[j][k]);j-=1
            if parent<0:break
            if j<0 or len(intersections[j])==0:break
            k=parent;c=costs[j]
    return selected,picked_s,restart

def grid_reconstruction_scores(lon,lat,grid,old_b2):
    xs=all_crossings(lon,lat,grid)
    counts=np.array([len(q) for q in xs],int)
    first=np.array([q[0,0] if len(q) else np.nan for q in xs]);last=np.array([q[-1,0] if len(q) else np.nan for q in xs])
    dp,ps,restart=continuous_arc_selector(xs,grid)
    nearest=np.array([float(np.min(np.abs(q[:,0]-v))*111.2) if len(q) and np.isfinite(v) else np.nan for q,v in zip(xs,old_b2)])
    spans=np.array([float(np.ptp(q[:,0])*111.2) if len(q)>=2 else np.nan for q in xs])
    spread=spans[np.isfinite(spans)]
    # 候选纬度属于真实线段交点，因此局地 off-axis 距离近零；
    # 这不能作为模型预测改善的证据，需再量化跳跃、不唯一性和分支歧义。
    def jump95(x):
        z=np.abs(np.diff(x))*111.2
        z=z[np.isfinite(z)]
        return float(np.percentile(z,95)) if len(z) else np.nan
    old_near=nearest[np.isfinite(nearest)]
    old_multi=np.array([v for v,k in zip(nearest,counts) if k>1 and np.isfinite(v)])
    valid=counts>0
    metrics={
        '经线格点数':len(grid),'无交点比例':float(np.mean(counts==0)),
        '多交点比例':float(np.mean(counts>=2)),
        '多交点平均跨度_km':float(np.mean(spread)) if len(spread) else 0.,
        '多交点跨度P95_km':float(np.percentile(spread,95)) if len(spread) else 0.,
        '旧表示距任一真实交点平均_km':float(np.mean(old_near)) if len(old_near) else np.nan,
        '旧表示距任一真实交点P95_km':float(np.percentile(old_near,95)) if len(old_near) else np.nan,
        '旧表示偏离真实交点超过5km比例':float(np.mean(nearest[valid]>5.0)) if valid.any() else np.nan,
        '旧表示多交点位置偏差均值_km':float(np.mean(old_multi)) if len(old_multi) else np.nan,
        '旧表示沿经度逐格纬度跳跃P95_km':jump95(old_b2),
        '首次交点逐格纬度跳跃P95_km':jump95(first),
        '末次交点逐格纬度跳跃P95_km':jump95(last),
        '连续弧长DP逐格纬度跳跃P95_km':jump95(dp),
        '连续弧长DP重启段数':restart,
        '连续DP与旧表示平均绝对差_km':float(np.mean(np.abs(dp[valid]-old_b2[valid])*111.2)) if valid.any() else np.nan,
        '最大多交点数':int(counts.max()),
    }
    return metrics,{'grid':grid,'old':old_b2,'first':first,'last':last,'dp':dp,'counts':counts,'spans':spans,'nearest':nearest,'dp_arc_s':ps}

# 本单元无需真实用户数据的几何单元测试
sx=np.linspace(121,146,400);sy=24+2*np.sin((sx-121)/25*5)
sg=np.array([122.,125.,130.,140.,145.])
xs=all_crossings(sx,sy,sg)
assert all(len(z)==1 for z in xs)
val,_,_=continuous_arc_selector(xs,sg)
assert np.all(np.isfinite(val))
assert np.max(np.abs(val-np.array([a[0,0] for a in xs])))<1e-8
mx=np.array([121.,124.,127.,125.,123.5,126.,130.,146.]);my=np.array([22.,24.,26.,29.,30.,31.,33.,36.])
assert max(len(x) for x in all_crossings(mx,my,np.arange(123,129,0.5)))>=2
print('[R55 GEOMETRY UNIT TEST] PASS: unique / multiple crossings, forward arclength DP')

In [ ]:
# ===== 6. 自动归档工具（先定义，在模型训练之前；失败也能独立执行） =====
import zipfile,traceback,hashlib
from IPython.display import FileLink,display,Markdown

def r56_archive(stage='partial',exc=None):
    REP.mkdir(parents=True,exist_ok=True)
    state={'experiment':'R5.6-geometric-representation-pilot','stage':stage,
           'timestamp_utc':pd.Timestamp.utcnow().isoformat(),'exception':exc,'config':asdict(cfg),
           'frozen_dataset':str(DS),'frozen_dataset_size':DS.stat().st_size if DS.exists() else None}
    (REP/'R56_阶段状态.json').write_text(json.dumps(state,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
    permitted={'.csv','.json','.md','.txt','.png','.pdf','.log','.yaml','.yml','.svg'}
    files=[]
    for p in OUT.rglob('*'):
        if not p.is_file() or p.is_symlink():continue
        rel=str(p.relative_to(OUT))
        if p.stat().st_size>cfg.ARCHIVE_MAX_FILE_MB*1024**2:continue
        if ('权重' in rel or p.suffix in ('.pt','.pth')) and not cfg.ARCHIVE_INCLUDE_WEIGHTS:continue
        if ('预测缓存' in rel or '派生表示' in rel or p.suffix=='.npz') and not cfg.ARCHIVE_INCLUDE_CACHE:continue
        if p.suffix.lower() not in permitted and p.suffix not in (('.pt','.pth') if cfg.ARCHIVE_INCLUDE_WEIGHTS else ()) and p.suffix!='.npz':continue
        if p.name=='R56_归档文件清单.csv':continue
        files.append(p)
    files=sorted(set(files))
    manifest=[{'相对路径':str(p.relative_to(OUT)),'字节数':p.stat().st_size,'SHA256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in files]
    mf=REP/'R56_归档文件清单.csv';pd.DataFrame(manifest).to_csv(mf,index=False,encoding='utf-8-sig')
    with zipfile.ZipFile(ARCHIVE_PATH,'w',zipfile.ZIP_DEFLATED,compresslevel=5,allowZip64=True) as z:
        for p in files:z.write(p,arcname='R56_结果/'+p.relative_to(OUT).as_posix())
        z.write(mf,arcname='R56_结果/报告/'+mf.name)
        bad=z.testzip()
        if bad:raise RuntimeError('ZIP校验错误 '+str(bad))
    digest=hashlib.sha256(ARCHIVE_PATH.read_bytes()).hexdigest()
    ARCHIVE_SHA.write_text(f'{digest}  {ARCHIVE_PATH.name}\n',encoding='utf-8')
    print('[R56 ARCHIVE PASS]',stage,ARCHIVE_PATH,'SHA256',digest,'items',len(files))
    display(FileLink(str(ARCHIVE_PATH)))
    print('Windows: D:\\KuroshioDaily1993_2025\\'+ARCHIVE_PATH.name)
    return ARCHIVE_PATH

print('[ARCHIVE READY] 即使训练未通过，也可单独调用 r56_archive("not_qualified").')

In [ ]:
# ===== 7. 生成独立候选表示（只修改B2；不覆盖原始NPZ） =====
A2_ORIGINAL=A2.copy();F2_ORIGINAL=F2.copy()
REAL_DATES=dates.copy()
DP_CACHE=DERIVED/'R56_B2_DP_AllDays.npz'
meta_path=DERIVED/'R56_候选构建签名.json'
# 资产签名阻止静默复用不同版本数据
track_files=sorted(RAW_TRACK_DIR.glob('*.npz'))
track_dates={p.stem:p for p in track_files}
sign_payload={'ds_name':DS.name,'ds_size':DS.stat().st_size,'ds_mtime_ns':DS.stat().st_mtime_ns,
    'tracks_n':len(track_files),'tracks_total_bytes':sum(p.stat().st_size for p in track_files),
    'r55_selector_version':'R55-arc-monotonic-v1','lon_n':int(len(lon2))}
sign=hashlib.sha256(json.dumps(sign_payload,sort_keys=True).encode()).hexdigest()
reuse=False
if DP_CACHE.exists() and meta_path.exists() and not cfg.REBUILD_DP:
    try:
        mm=json.loads(meta_path.read_text())
        with np.load(DP_CACHE,allow_pickle=False) as z:
            bc=np.asarray(z['b2_dp'],np.float32);dc=np.asarray(z['dates']).astype(str)
        if mm.get('signature')==sign and bc.shape==(len(dates),len(lon2)) and np.array_equal(dc,dates.strftime('%Y-%m-%d').to_numpy()):reuse=True
    except Exception as e:print('[DP cache ignored]',e)
if reuse:
    B2_DP=bc;print('[DP CACHE RESUME]',DP_CACHE)
else:
    B2_DP=np.empty((len(dates),len(lon2)),np.float32)
    rows=[];fail=[]
    for ix,dt in enumerate(dates):
        key=dt.strftime('%Y%m%d');p=track_dates.get(key)
        if p is None:
            B2_DP[ix]=b2[ix]; fail.append((str(dt.date()),'raw_track_not_found'))
        else:
            try:
                with np.load(p,allow_pickle=False) as zz:
                    lon=np.asarray(zz['lon'],float);lat=np.asarray(zz['lat'],float)
                crosses=all_crossings(lon,lat,lon2)
                arr,ss,nrestart=continuous_arc_selector(crosses,lon2)
                if not np.isfinite(arr).all():
                    B2_DP[ix]=b2[ix];fail.append((str(dt.date()),f'nonfinite crossings={int((~np.isfinite(arr)).sum())}'))
                else:
                    B2_DP[ix]=arr.astype(np.float32)
                    rows.append({'日期':str(dt.date()),'连续弧长DP重启段数':int(nrestart),
                      '原始与DP平均差_km':float(np.mean(np.abs(arr-b2[ix])*111.2)),
                      '原始与DP最大差_km':float(np.max(np.abs(arr-b2[ix])*111.2)),
                      '多交点经线比例':float(np.mean([len(c)>1 for c in crosses]))})
            except Exception as e:
                B2_DP[ix]=b2[ix];fail.append((str(dt.date()),repr(e)))
        if (ix+1)%cfg.DP_REPORT_EVERY==0:
            print(f'[R56 DP] {ix+1}/{len(dates)} fallbacks={len(fail)}')
    pd.DataFrame(rows).to_csv(DATA/'R56_01_DP重建逐日审计.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(fail,columns=['日期','原因']).to_csv(DATA/'R56_01_DP回退记录.csv',index=False,encoding='utf-8-sig')
    np.savez_compressed(DP_CACHE,b2_dp=B2_DP,dates=dates.strftime('%Y-%m-%d').to_numpy())
    meta_path.write_text(json.dumps({'signature':sign,'payload':sign_payload,'fallback_days':len(fail)},indent=2,ensure_ascii=False),encoding='utf-8')
# 逐日时间稳定性：只比较相邻真实有效日；禁止平滑时间序列以读取未来信息
pos2=cal.get_indexer(dates)
consecutive=np.diff(pos2)==1
oldstep=np.sqrt(np.mean((b2[1:]-b2[:-1])**2,axis=1))*111.2
newstep=np.sqrt(np.mean((B2_DP[1:]-B2_DP[:-1])**2,axis=1))*111.2
old95=float(np.percentile(oldstep[consecutive],95));new95=float(np.percentile(newstep[consecutive],95))
ratio=new95/max(old95,1e-6)
missing_count=int(json.loads(meta_path.read_text()).get('fallback_days',0))
geom_gate=bool(missing_count/len(dates)<=cfg.MAX_MISSING_DP_DAY_FRAC and ratio<=cfg.MAX_DP_TEMPORAL_P95_RATIO)
geom_stats={'days':len(dates),'fallback_days':missing_count,'fallback_frac':missing_count/len(dates),
   'old_day_to_day_rms_p95_km':old95,'dp_day_to_day_rms_p95_km':new95,
   'dp_to_old_temporal_p95_ratio':ratio,'max_ratio_gate':cfg.MAX_DP_TEMPORAL_P95_RATIO,
   'max_fallback_frac_gate':cfg.MAX_MISSING_DP_DAY_FRAC,'geometry_pilot_gate':geom_gate,
   'note':'该门槛只约束候选表示物理与时序可用性，不意味着预测提升'}
(REP/'R56_01_几何资格判定.json').write_text(json.dumps(geom_stats,indent=2,ensure_ascii=False),encoding='utf-8')
print('[GEOMETRY PILOT]',geom_stats)
# 防止任意几何失败仍进入训练
r56_archive('geometry_preflight_complete')

In [ ]:
# ===== 5. 每折 train-only 标准化 + 自适应 99% EOF =====
# 使用该fold训练期的独立日记录，而不是重叠180d窗口重复叠加做SVD；内存恒定在 O(feature²)
class DailyEOF:
    def fit(self,obs,target=0.99):
        obs=np.asarray(obs,np.float64)
        self.mu=np.mean(obs,axis=0); self.sd=np.std(obs,axis=0); self.sd[self.sd<1e-6]=1.
        z=(obs-self.mu)/self.sd
        cov=(z.T@z)/max(1,len(z)-1)
        eig,vec=np.linalg.eigh(cov)
        idx=np.argsort(eig)[::-1];eig=np.maximum(eig[idx],0);vec=vec[:,idx]
        rat=np.cumsum(eig)/(eig.sum()+1e-12); n=int(np.searchsorted(rat,target)+1)
        self.V=np.asarray(vec[:,:n],np.float32);self.explained=float(rat[n-1]);self.n=n
        return self
    def pc(self,x): return (((np.asarray(x)-self.mu)/self.sd)@self.V).astype(np.float32)
    def invert(self,pc): return ((np.asarray(pc)@self.V.T)*self.sd+self.mu).astype(np.float32)
    def z(self,x): return ((np.asarray(x)-self.mu)/self.sd).astype(np.float32)

def make_fold_data(f):
    train_days=(cal< f['train_cut']) & valid
    pp1=DailyEOF().fit(A1[train_days],cfg.EOF_VARIANCE_TARGET)
    pp2=DailyEOF().fit(A2[train_days],cfg.EOF_VARIANCE_TARGET)
    for pp in (pp1,pp2):
        if pp.explained<cfg.EOF_VARIANCE_TARGET-1e-6: raise RuntimeError('EOF 99%训练方差Gate未通过')
    # approx memory: ~17k days * 2*(31+277) floats; never expands 180 day windows
    z1=pp1.z(F1);z2=pp2.z(F2)
    pcs1=pp1.pc(F1);pcs2=pp2.pc(F2)
    ppath=CKPT/ACTIVE_REP/f'第{f["fold"]}折_{f["year"]}年'; ppath.mkdir(exist_ok=True,parents=True)
    np.savez_compressed(ppath/'训练期预处理参数.npz',
        b1_mu=pp1.mu,b1_sd=pp1.sd,b1_V=pp1.V,b2_mu=pp2.mu,b2_sd=pp2.sd,b2_V=pp2.V,
        branch1_lat=lat1,branch2_lon=lon2,train_label_cutoff=str(f['train_cut'].date()))
    return {'pp1':pp1,'pp2':pp2,'z1':z1,'z2':z2,'p1':pcs1,'p2':pcs2,'path':ppath}

# 只预审一折，防止全局重复保留大数组
_testFD=make_fold_data(FOLDS[0]);print('fold1 EOF modes:',_testFD['pp1'].n,_testFD['pp2'].n,'explained:',_testFD['pp1'].explained,_testFD['pp2'].explained)
del _testFD;gc.collect()
print('[TRAIN-ONLY EOF 99%] PASS')

In [ ]:
# ===== 6. 动态加载数据集，训练时每个batch只展开当前180天窗口 =====
class CausalWindowDataset(Dataset):
    def __init__(self,issues,scaled,pcs):
        self.issues=np.asarray(issues,np.int32);self.scaled=scaled;self.pcs=pcs
    def __len__(self): return len(self.issues)
    def __getitem__(self,j):
        i=int(self.issues[j]);z=self.scaled[i-H+1:i+1]
        dz=np.empty_like(z);dz[0]=0;dz[1:]=z[1:]-z[:-1]
        x=np.concatenate([z,dz],axis=-1)
        y=np.stack([self.pcs[i+d] for d in cfg.LEADS])
        return torch.from_numpy(x.copy()),torch.from_numpy(y.copy())

class StepEncoder(nn.Module):
    # 与旧原稿的per-step TCN/Conv编码保持一致：输入路径特征卷积，然后LSTM承担时间建模
    def __init__(self,feature,branch,kind='tcn'):
        super().__init__();self.kind=kind;self.drop=nn.Dropout(cfg.DROPOUT)
        if kind=='tcn':
            channels=16 if branch==1 else 32;kernel=3 if branch==1 else 5
            self.conv=nn.Conv1d(1,channels,kernel,dilation=2)
            self.bn=nn.BatchNorm1d(channels)
            conv_len=feature+2-2*(kernel-1)
            self.fc=nn.Linear(channels*conv_len,32 if branch==1 else 64)
            self.out_dim=self.fc.out_features
        elif kind=='conv':
            c1=16 if branch==1 else 32;c2=32 if branch==1 else 64
            self.c1=nn.Conv1d(1,c1,3,padding=1);self.c2=nn.Conv1d(c1,c2,3,padding=1)
            self.out_dim=c2*feature
        else: raise ValueError(kind)
    def forward(self,x):
        B,T,F=x.shape; z=x.reshape(B*T,1,F)
        if self.kind=='tcn':
            z=torch.nn.functional.pad(z,(2,0)) # 与R3原架构一致的左侧填充
            z=self.drop(torch.relu(self.bn(self.conv(z))))
            z=self.drop(torch.relu(self.fc(z.flatten(1))))
        else:
            z=self.drop(torch.relu(self.c1(z)));z=self.drop(torch.relu(self.c2(z)));z=z.flatten(1)
        return z.reshape(B,T,-1)

class Hybrid(nn.Module):
    def __init__(self,feature,pc_dim,branch,kind='tcn',recur='lstm'):
        super().__init__();self.enc=StepEncoder(feature,branch,kind)
        h=32 if branch==1 else 64
        R=nn.GRU if recur=='gru' else nn.LSTM
        self.rnn=R(self.enc.out_dim,h,batch_first=True)
        self.fc=nn.Linear(h,3*pc_dim);self.pc_dim=pc_dim
    def forward(self,x):
        h,_=self.rnn(self.enc(x));return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

class MatchedLSTM(nn.Module):
    def __init__(self,feature,pc_dim,hidden):
        super().__init__();self.rnn=nn.LSTM(feature,hidden,batch_first=True)
        self.fc=nn.Linear(hidden,3*pc_dim);self.pc_dim=pc_dim
    def forward(self,x):
        h,_=self.rnn(x);return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

def nparams(m): return sum(p.numel() for p in m.parameters())
def match_lstm(feature,pc_dim,branch):
    target=nparams(Hybrid(feature,pc_dim,branch))
    # Pytorch single-layer LSTM: 4h(feature+h+2) + 3pc*h + 3pc
    opts=range(8,1025)
    h=min(opts,key=lambda hh:abs(4*hh*(feature+hh+2)+(3*pc_dim)*(hh+1)-target))
    actual=4*h*(feature+h+2)+(3*pc_dim)*(h+1)
    if abs(actual-target)/target>0.08: raise RuntimeError('Matched-LSTM参数量相对差>8%，不满足公平基线')
    return h,target,actual

def make_net(name,branch,fd):
    feature=fd[f'z{branch}'].shape[1]*2;pdim=fd[f'pp{branch}'].n
    if name=='Matched-LSTM':
        h,_,_=match_lstm(feature,pdim,branch);return MatchedLSTM(feature,pdim,h)
    if name=='EOF-TCN-LSTM': return Hybrid(feature,pdim,branch,'tcn','lstm')
    if name=='EOF-Conv-LSTM': return Hybrid(feature,pdim,branch,'conv','lstm')
    if name=='EOF-Conv-GRU': return Hybrid(feature,pdim,branch,'conv','gru')
    if name=='EOF-TCN-GRU': return Hybrid(feature,pdim,branch,'tcn','gru')
    raise ValueError(name)




# ===== R5.3：路径坐标/一阶差分分为两个空间输入通道 =====
class ChannelSeparatedEncoder(nn.Module):
    """单时刻空间卷积；输入[B,T,2*M] => Conv1d [B*T,2,M]."""
    def __init__(self,feature,branch):
        super().__init__()
        assert feature%2==0, '需要[x, delta]等长拼接输入'
        self.raw_dim=feature//2
        base_c=16 if branch==1 else 32
        k=3 if branch==1 else 5
        channels=base_c*2
        # 正确的对称空间填充：dilation=2 -> padding=k-1，输出仍为M
        self.conv=nn.Conv1d(2,channels,kernel_size=k,dilation=2,padding=k-1)
        self.bn=nn.BatchNorm1d(channels)
        self.fc=nn.Linear(channels*self.raw_dim,32 if branch==1 else 64)
        self.out_dim=self.fc.out_features
        self.drop=nn.Dropout(cfg.DROPOUT)
    def channelize(self,x):
        B,T,F=x.shape
        if F!=self.raw_dim*2: raise RuntimeError('R53特征维不匹配')
        return torch.stack((x[:,:,:self.raw_dim],x[:,:,self.raw_dim:]),dim=2).reshape(B*T,2,self.raw_dim)
    def forward(self,x):
        B,T,F=x.shape
        z=self.channelize(x)
        z=self.drop(torch.relu(self.bn(self.conv(z))))
        if z.shape[2]!=self.raw_dim:raise RuntimeError('R53 spatial-conv输出宽度发生变化')
        z=self.drop(torch.relu(self.fc(z.flatten(1))))
        return z.reshape(B,T,-1)

class ChannelHybrid(nn.Module):
    def __init__(self,feature,pc_dim,branch,projection=None,use_pc_skip=False):
        super().__init__()
        self.enc=ChannelSeparatedEncoder(feature,branch)
        self.use_pc_skip=bool(use_pc_skip)
        self.raw_dim=feature//2
        self.pc_dim=pc_dim
        if self.use_pc_skip:
            self.register_buffer('V',torch.as_tensor(projection,dtype=torch.float32))
        hidden=32 if branch==1 else 64
        in_dim=self.enc.out_dim+(2*pc_dim if self.use_pc_skip else 0)
        self.rnn=nn.LSTM(in_dim,hidden,batch_first=True)
        self.fc=nn.Linear(hidden,3*pc_dim)
    def eof_skip(self,x):
        if not self.use_pc_skip: raise RuntimeError('no PC skip in this model')
        return torch.cat((x[:,:,:self.raw_dim]@self.V,x[:,:,self.raw_dim:]@self.V),dim=-1)
    def forward(self,x):
        z=self.enc(x)
        if self.use_pc_skip: z=torch.cat((z,self.eof_skip(x)),dim=-1)
        h,_=self.rnn(z)
        return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

R53_CANDIDATES=('R53-ChannelConv','R53-ChannelConv-PCConcat')
R53_BASELINES=('Matched-LSTM','EOF-TCN-LSTM','R53-CapacityMatchedLSTM')
R53_ALL=R53_BASELINES+R53_CANDIDATES

def make_r53_model(name,branch,fd):
    feature=fd[f'z{branch}'].shape[1]*2
    pdim=fd[f'pp{branch}'].n
    if name in ('Matched-LSTM','EOF-TCN-LSTM'):
        return make_net(name,branch,fd)
    if name=='R53-CapacityMatchedLSTM':
        ref=ChannelHybrid(feature,pdim,branch,fd[f'pp{branch}'].V,use_pc_skip=True)
        h,_,_=matched_hidden_for_target(feature,pdim,nparams(ref))
        return MatchedLSTM(feature,pdim,h)
    if name=='R53-ChannelConv':
        return ChannelHybrid(feature,pdim,branch)
    if name=='R53-ChannelConv-PCConcat':
        return ChannelHybrid(feature,pdim,branch,fd[f'pp{branch}'].V,use_pc_skip=True)
    raise ValueError(name)

def matched_hidden_for_target(feature,pdim,target):
    best=min(range(8,1025),key=lambda h:abs(nparams(MatchedLSTM(feature,pdim,h))-target))
    actual=nparams(MatchedLSTM(feature,pdim,best))
    ratio=abs(actual-target)/max(target,1)
    if ratio>0.08:raise RuntimeError(f'参数匹配误差过大：{ratio:.2%}')
    return best,actual,ratio

def model_loss(pred,target,fd,branch,name):
    # 与R5.2完全相同的训练目标：只用PC-MSE做结构归因。
    return torch.nn.functional.mse_loss(pred,target)

In [ ]:
# ===== 8. 物理公里误差：局地投影+最近点相邻线段的双向正交路径距离 =====
# 对于仅122-145E的分支2，重建得到共同定义域；不是对每一点简单经纬度平方误差。
# 下方是双向point-to-polyline近似cross-track，使用KDTree近邻三顶点及其邻接segments。
def merge_path(branch1,branch2):
    a=np.asarray(branch1,float);b=np.asarray(branch2,float)
    candidates=np.flatnonzero(lon2>a[-1]);
    k=int(candidates[np.argmin((lon2[candidates]-a[-1])**2+(b[candidates]-lat1[-1])**2)]) if len(candidates) else len(lon2)
    xs=np.r_[a,lon2[k:]];ys=np.r_[lat1,b[k:]]
    good=np.isfinite(xs)&np.isfinite(ys)
    return xs[good],ys[good]

def one_direction(A,B):
    # A,B: 2d coordinates (km); approximate nearest polyline segment for each A point
    if len(A)<2 or len(B)<2: return np.array([np.nan])
    tree=cKDTree(B);_,knn=tree.query(A,k=min(3,len(B)))
    knn=np.asarray(knn);knn=knn[:,None] if knn.ndim==1 else knn
    ix=np.unique(np.clip(np.concatenate([knn.ravel()-1,knn.ravel()]),0,len(B)-2))
    # segment shortlist globally may grow, but polynomial per path length (~300) and is vectorized
    v=B[ix+1]-B[ix];w=A[:,None,:]-B[ix][None,:,:]
    den=np.maximum(np.sum(v*v,axis=1),1e-12)
    t=np.clip(np.sum(w*v[None,:,:],axis=2)/den[None,:],0,1)
    q=B[ix][None,:,:]+t[:,:,None]*v[None,:,:]
    d=np.sqrt(np.sum((A[:,None,:]-q)**2,axis=2))
    return d.min(axis=1)

def errors_for_pair(pred1,pred2,truth1,truth2):
    x,y=merge_path(pred1,pred2);xt,yt=merge_path(truth1,truth2)
    lat0=np.mean(np.r_[y,yt]);lon0=np.mean(np.r_[x,xt]);c=np.cos(np.deg2rad(lat0))
    A=np.column_stack([(x-lon0)*111.32*c,(y-lat0)*111.32]);B=np.column_stack([(xt-lon0)*111.32*c,(yt-lat0)*111.32])
    d=np.r_[one_direction(A,B),one_direction(B,A)]
    return float(np.mean(d**2)),float(np.mean(d)),float(np.percentile(d,90))

def record_errors(fold,model,seed,issues,p1,p2):
    out=[]
    for n,i in enumerate(issues):
        for k,lead in enumerate(cfg.LEADS):
            y1=A1[i+lead];y2=A2[i+lead]
            mse,mean,p90=errors_for_pair(p1[n,k],p2[n,k],y1,y2)
            out.append({'折次':fold['fold'],'测试年份':fold['year'],'模型':model,'种子':seed,'起报日期':str(cal[i].date()),'目标日期':str(cal[i+lead].date()),'提前期_天':int(lead),
                        '横向路径误差MSE_km2':mse,'横向路径平均误差_km':mean,'横向路径P90_km':p90,
                        '分支1经度RMSE_度':float(np.sqrt(np.mean((p1[n,k]-y1)**2))),
                        '分支2纬度RMSE_度':float(np.sqrt(np.mean((p2[n,k]-y2)**2)))})
    return pd.DataFrame(out)

In [ ]:
# ===== 9. R5 训练封装、物理评价、可恢复 checkpoints =====
from contextlib import nullcontext

def r5_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def run_predict(net, issues, fd, branch):
    ds=CausalWindowDataset(issues,fd[f'z{branch}'],fd[f'p{branch}'])
    ld=DataLoader(ds,batch_size=cfg.BATCH_SIZE,shuffle=False,num_workers=0)
    arr=[];net.eval()
    with torch.no_grad():
        for xx,_ in ld: arr.append(net(xx.to(DEV)).float().cpu().numpy())
    return np.concatenate(arr,axis=0)

def fit_r5_branch(split, fd, name, seed, branch, root_dir, max_epochs=None, patience=None):
    max_epochs=cfg.PILOT_EPOCHS if max_epochs is None else max_epochs
    patience=cfg.PILOT_PATIENCE if patience is None else patience
    folder=root_dir/ACTIVE_REP/f'QA_{split["year"]-1}'/name/f'seed_{seed}'
    folder.mkdir(parents=True,exist_ok=True)
    weight=folder/f'branch{branch}_best.pt'; historyfile=folder/f'branch{branch}_history.csv'
    latest=folder/f'branch{branch}_latest.pt'
    # best checkpoint 附带实验合同签名；仅同一fold/样本划分/网络版本可复用
    contract={'name':name,'seed':seed,'branch':branch,'tr_n':int(len(split['train'])),
              'val_n':int(len(split['val'])),'tr_last':int(split['train'][-1]),
              'val_first':int(split['val'][0]),'val_last':int(split['val'][-1]),'train_first':int(split['train'][0]),'max_epochs':max_epochs,
              'feature_dims':[fd['z1'].shape[1],fd['z2'].shape[1]],'version':'R5.6-geometry-v1','rep':ACTIVE_REP,'dataset_signature':sign}
    contract_id=hashlib.sha256(json.dumps(contract,sort_keys=True).encode()).hexdigest()
    r5_seed(seed)
    net=make_r53_model(name,branch,fd).to(DEV)
    if weight.exists() and historyfile.exists() and not cfg.FORCE_TRAIN:
        ck=torch.load(weight,map_location='cpu',weights_only=True)
        if ck.get('contract_id')==contract_id:
            net.load_state_dict(ck['state']);return net,pd.read_csv(historyfile)
    ds_tr=CausalWindowDataset(split['train'],fd[f'z{branch}'],fd[f'p{branch}'])
    ds_va=CausalWindowDataset(split['val'],fd[f'z{branch}'],fd[f'p{branch}'])
    ld_tr=DataLoader(ds_tr,batch_size=cfg.BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=True)
    ld_va=DataLoader(ds_va,batch_size=cfg.BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
    opt=torch.optim.AdamW(net.parameters(),lr=cfg.LR,weight_decay=cfg.WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,factor=0.5,patience=4)
    amp=bool(cfg.USE_AMP)
    scaler=torch.amp.GradScaler('cuda',enabled=amp)
    start_epoch=1;best=float('inf');bestep=0;bad=0;hist=[]
    if latest.exists() and not cfg.FORCE_TRAIN:
        try:
            ck=torch.load(latest,map_location='cpu',weights_only=True)
            if ck.get('contract_id')==contract_id:
                net.load_state_dict(ck['state']);opt.load_state_dict(ck['opt']);scheduler.load_state_dict(ck['scheduler']);scaler.load_state_dict(ck['scaler'])
                start_epoch=int(ck['epoch'])+1;best=float(ck['best']);bestep=int(ck['bestep']);bad=int(ck['bad']);hist=ck['hist']
                print('[RESUME]',folder,'epoch',start_epoch)
        except Exception as e:print('[resume invalid]',e)
    for ep in range(start_epoch,max_epochs+1):
        net.train();ts=tn=0
        for x,y in ld_tr:
            x=x.to(DEV,non_blocking=True);y=y.to(DEV,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',enabled=amp):
                pred=net(x)
                loss=model_loss(pred.float(),y.float(),fd,branch,name)
            if not torch.isfinite(loss):raise RuntimeError(f'{name} B{branch}: non-finite loss')
            scaler.scale(loss).backward();scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(),1.0)
            scaler.step(opt);scaler.update();ts+=float(loss.detach())*len(x);tn+=len(x)
        net.eval();vs=vn=0
        with torch.no_grad():
            for x,y in ld_va:
                x=x.to(DEV);y=y.to(DEV)
                with torch.autocast('cuda',enabled=amp):
                    p=net(x);loss=model_loss(p.float(),y.float(),fd,branch,name)
                vs+=float(loss.detach())*len(x);vn+=len(x)
        val=vs/max(1,vn);scheduler.step(val)
        hist.append({'epoch':ep,'train_loss':ts/max(1,tn),'val_loss':val,'lr':opt.param_groups[0]['lr']})
        if val<best-1e-7:
            best=val;bestep=ep;bad=0
            torch.save({'state':{k:v.detach().cpu() for k,v in net.state_dict().items()},'contract_id':contract_id,'epoch':ep,'best_val':best},weight)
        else:bad+=1
        if ep%5==0 or bad>=patience or ep==max_epochs:
            torch.save({'state':{k:v.detach().cpu() for k,v in net.state_dict().items()},'opt':opt.state_dict(),
                        'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'epoch':ep,'best':best,
                        'bestep':bestep,'bad':bad,'hist':hist,'contract_id':contract_id},latest)
        if ep==1 or ep%10==0 or bad>=patience:print(f'{name} B{branch} seed{seed} ep{ep} train={hist[-1]["train_loss"]:.5f} val={val:.5f}')
        if bad>=patience:break
    pd.DataFrame(hist).to_csv(historyfile,index=False,encoding='utf-8-sig')
    if latest.exists():latest.unlink()
    ck=torch.load(weight,map_location='cpu',weights_only=True)
    net.load_state_dict(ck['state']);net.eval()
    return net,pd.DataFrame(hist)

def evaluate_issues(f,name,seed,fd,issues,bmodel1,bmodel2):
    return record_errors(f,name,seed,issues,bmodel1,bmodel2)

def get_predictions(name,split,fd,seed,folder,max_epochs=None,patience=None):
    branches=[]
    for branch in (1,2):
        net,hist=fit_r5_branch(split,fd,name,seed,branch,folder,max_epochs,patience)
        pr=run_predict(net,split.get('qa',split.get('test')),fd,branch)
        branches.append(fd[f'pp{branch}'].invert(pr))
        del net,pr;torch.cuda.empty_cache()
    return branches

In [ ]:
# ===== 11. 内层早停与独立QA时间段：不能拿QA来早停或改训练损失 =====
def make_pilot_fold(f):
    year=f['year'];inner_year=year-2; qa_year=year-1
    qa_start=pd.Timestamp(qa_year,1,1)
    early_start=pd.Timestamp(inner_year,1,1)
    train_cut=early_start-pd.Timedelta(days=cfg.PURGE_DAYS)
    train=ISS[cal[ISS+L]<train_cut]
    # early-stop inner year issue，最后标签必须在QA起报之前
    early=ISS[(cal[ISS]>=early_start)&(cal[ISS]<qa_start)&(cal[ISS+L]<qa_start-pd.Timedelta(days=cfg.PURGE_DAYS))]
    qa=f['val']
    if len(train)<cfg.MIN_TRAIN or len(early)<cfg.MIN_VAL or len(qa)<cfg.MIN_VAL:
        raise RuntimeError(f'Pilot split {year} insufficient train={len(train)} early={len(early)} QA={len(qa)}')
    if not (cal[train+L].max()<cal[early].min()-pd.Timedelta(days=cfg.PURGE_DAYS) and
            cal[early+L].max()<cal[qa].min()-pd.Timedelta(days=cfg.PURGE_DAYS)):
        raise RuntimeError('Pilot temporal leakage')
    return {'fold':500+f['fold'],'year':f['year'],'train':train,'val':early,'qa':qa,'train_cut':train_cut}

PILOTS=[make_pilot_fold(next(f for f in FOLDS if f['year']==yr)) for yr in cfg.PILOT_TEST_YEARS]
pp=[]
for f in PILOTS:
    pp.append({'对应测试年份':f['year'],'训练样本':len(f['train']),'内层早停样本':len(f['val']),
               '独立机制资格样本':len(f['qa']), '训练最晚标签':str(cal[f['train']+L].max().date()),
               '早停最晚标签':str(cal[f['val']+L].max().date()),'QA最早起报':str(cal[f['qa']].min().date())})
pd.DataFrame(pp).to_csv(DATA/'R53_02_机制资格实验严格时间划分.csv',index=False,encoding='utf-8-sig')
print('[PILOT STRICT TEMPORAL QA] PASS');display(pd.DataFrame(pp))

In [ ]:
# ===== 12. 全链路小规模机制与时序预检 =====
def activate(rep):
    global ACTIVE_REP,A2,F2
    assert rep in ('Original','DP-B2')
    ACTIVE_REP=rep
    A2=A2_ORIGINAL.copy()
    if rep=='DP-B2':A2[pos2]=B2_DP
    F2=causal_fill(A2,cfg.MAX_INPUT_GAP)
    assert np.isfinite(F2[valid]).all()

for rep in ('Original','DP-B2'):
    activate(rep)
    fd=make_fold_data(PILOTS[0]);dims=[]
    for branch in (1,2):
        for name in ('Matched-LSTM','R53-CapacityMatchedLSTM','EOF-TCN-LSTM','R53-ChannelConv-PCConcat'):
            net=make_r53_model(name,branch,fd).to(DEV)
            x=torch.randn(2,H,fd[f'z{branch}'].shape[1]*2,device=DEV)
            y=net(x)
            assert y.shape==(2,3,fd[f'pp{branch}'].n) and bool(torch.isfinite(y).all())
            y.square().mean().backward()
            dims.append({'候选表示':rep,'模型':name,'分支':branch,'参数量':nparams(net),'EOF维数':fd[f'pp{branch}'].n})
            del net
    pd.DataFrame(dims).to_csv(DATA/f'R56_02_{rep}_模型参数量.csv',index=False,encoding='utf-8-sig')
    del fd;torch.cuda.empty_cache()
activate('Original')
print('[R56 MODEL-SHAPE+BACKWARD+CAUSAL SPLIT] PASS')

In [ ]:
# ===== 13. 固定双表示 × 固定三模型的小规模QA实验 =====
# 这次仅使用已经暴露的开发QA年份，不允许作为发表时的盲测泛化证据。
QA_MODELS=('Matched-LSTM','R53-CapacityMatchedLSTM','EOF-TCN-LSTM','R53-ChannelConv-PCConcat')
QA_REPS=('Original','DP-B2')
A2_FROZEN=A2_ORIGINAL.copy()

# 提前固定原始流轴抽样issue，所有表示和模型严格一致
RAW_SUBSETS={}
for f in PILOTS:
    ids=np.asarray(f['qa'],np.int32)
    picks=np.unique(np.linspace(0,len(ids)-1,min(len(ids),cfg.RAW_EVAL_ISSUES_PER_QA_YEAR),dtype=int))
    RAW_SUBSETS[f['year']-1]=set(map(int,ids[picks]))

def record_dual_reference(split,model,rep,seed,issues,p1,p2):
    out=[]
    for n,ii in enumerate(issues):
        ii=int(ii)
        for k,lead in enumerate(cfg.LEADS):
            j=ii+lead
            # 冻结真值：所有方法、表示共同的对照标准
            o=errors_for_pair(p1[n,k],p2[n,k],A1[j],A2_FROZEN[j])
            # 自身表示参考：只作诊断，不能拿它直接排名两个不同表示
            u=errors_for_pair(p1[n,k],p2[n,k],A1[j],A2[j])
            row={'QA年份':split['year']-1,'模型':model,'表示':rep,'种子':seed,
                '起报日期':str(cal[ii].date()),'目标日期':str(cal[j].date()),'提前期_天':int(lead),
                '共同冻结目标MSE_km2':o[0],'候选自身目标MSE_km2':u[0],
                '冻结目标平均距离_km':o[1],'DP相对旧目标差异_km':float(np.sqrt(np.mean((A2[j]-A2_FROZEN[j])**2))*111.2)}
            if ii in RAW_SUBSETS[split['year']-1]:
                p=RAW_TRACK_DIR/(str(cal[j].strftime('%Y%m%d'))+'.npz')
                if p.exists():
                    with np.load(p,allow_pickle=False) as zz:
                        raw_lon=np.asarray(zz['lon'],float);raw_lat=np.asarray(zz['lat'],float)
                    x,y=merge_path(p1[n,k],p2[n,k]);ok=np.isfinite(raw_lon)&np.isfinite(raw_lat)
                    if ok.sum()>=2 and len(x)>=2:
                        rx,ry=raw_lon[ok],raw_lat[ok]
                        lat0=float(np.mean(np.r_[y,ry]));lon0=float(np.mean(np.r_[x,rx]));c=np.cos(np.deg2rad(lat0))
                        xy=np.column_stack(((x-lon0)*111.32*c,(y-lat0)*111.32))
                        truth=np.column_stack(((rx-lon0)*111.32*c,(ry-lat0)*111.32))
                        # Symmetric track distance: sampled polyline vs original day's accepted tracking path
                        distances=np.r_[one_direction(xy,truth),one_direction(truth,xy)]
                        row['原始追踪流轴距离MSE_km2']=float(np.mean(distances*distances))
            out.append(row)
    return pd.DataFrame(out)

QA_PATH=DATA/'R56_03_双表示固定模型QA逐样本.csv'
QA_CACHE=OUT/'资格实验预测缓存';QA_CACHE.mkdir(exist_ok=True,parents=True)
R56_QA=None
if not geom_gate:
    warning='[R56 GEOMETRY GATE FAILED] DP 候选时间连续性/回退率不足，停止训练。只报告结果，不声称效果改善。'
    print(warning)
    (REP/'R56_资格实验阻断说明.txt').write_text(warning,encoding='utf-8')
    r56_archive('geometry_gate_failed')
else:
    rows=[]
    try:
        for rep in QA_REPS:
            activate(rep)
            for f in PILOTS:
                print('\n[R56 QA]',rep,'QA year',f['year']-1)
                fd=make_fold_data(f)
                try:
                    for name in QA_MODELS:
                        for seed in cfg.PILOT_SEEDS:
                            cf=QA_CACHE/f'{rep}_{f["year"]}_{name}_seed{seed}.npz'
                            if cf.exists() and not cfg.FORCE_TRAIN:
                                with np.load(cf,allow_pickle=False) as z:
                                    a=z['b1'];b=z['b2'];ix=z['issues'];saved_rep=str(z['representation'])
                                if saved_rep!=rep or not np.array_equal(ix,f['qa']):raise RuntimeError('QA prediction cache mismatch: '+str(cf))
                            else:
                                a,b=get_predictions(name,f,fd,seed,OUT/'资格实验权重',cfg.PILOT_EPOCHS,cfg.PILOT_PATIENCE)
                                np.savez_compressed(cf,b1=a,b2=b,issues=f['qa'],representation=rep)
                            row=record_dual_reference(f,name,rep,seed,f['qa'],a,b)
                            rows.append(row)
                            pd.concat(rows,ignore_index=True).to_csv(QA_PATH,index=False,encoding='utf-8-sig')
                finally:
                    del fd;gc.collect();torch.cuda.empty_cache()
        R56_QA=pd.concat(rows,ignore_index=True)
        r56_archive('qa_prediction_complete')
    except Exception as e:
        (REP/'R56_运行异常.txt').write_text(traceback.format_exc(),encoding='utf-8')
        r56_archive('training_exception',traceback.format_exc())
        raise

In [ ]:
# ===== 14. 统一原冻结真值的成对评分、年份分层与准入决策 =====
if R56_QA is None and QA_PATH.exists():R56_QA=pd.read_csv(QA_PATH)
if R56_QA is None or len(R56_QA)==0:
    result={'qualified':False,'reason':'geometry_gate_failed_or_no_completed_predictions', 'geometry_gate':geom_stats}
    (REP/'R56_最终交接.json').write_text(json.dumps(result,indent=2,ensure_ascii=False),encoding='utf-8')
    r56_archive('completed_without_training')
else:
    # 对同一种子、同一个年份、同一个预测提前期先做RMSE；然后等年份等种子平均
    run=R56_QA.groupby(['QA年份','模型','表示','种子','提前期_天'],as_index=False).agg(
      公共冻结RMSE_km=('共同冻结目标MSE_km2',lambda x:float(np.sqrt(np.mean(x)))),
      自身目标RMSE_km=('候选自身目标MSE_km2',lambda x:float(np.sqrt(np.mean(x)))),
      原始流轴RMSE_km=('原始追踪流轴距离MSE_km2',lambda x:float(np.sqrt(np.mean(x.dropna()))) if x.notna().any() else np.nan))
    run.to_csv(DATA/'R56_04_模型种子年份提前期指标.csv',index=False,encoding='utf-8-sig')
    mean=run.groupby(['模型','表示','提前期_天'],as_index=False)[['公共冻结RMSE_km','自身目标RMSE_km','原始流轴RMSE_km']].mean()
    mean.to_csv(DATA/'R56_04_各表示模型公平评价汇总.csv',index=False,encoding='utf-8-sig')
    # 完全成对比较同样issue/test years/seed/lead；对原冻结目标与原始轨迹两种公共真值分别审计
    pivot=run.pivot_table(index=['QA年份','模型','种子','提前期_天'],columns='表示',values=['公共冻结RMSE_km','原始流轴RMSE_km'])
    pair=[]
    for idx,rr in pivot.iterrows():
        year,name,seed,lead=idx
        pair.append({'QA年份':year,'模型':name,'种子':seed,'提前期_天':lead,
          'DP相对Original_公共冻结改善_km':float(rr[('公共冻结RMSE_km','Original')]-rr[('公共冻结RMSE_km','DP-B2')]),
          'DP相对Original_原始流轴改善_km':float(rr[('原始流轴RMSE_km','Original')]-rr[('原始流轴RMSE_km','DP-B2')])})
    pair=pd.DataFrame(pair)
    pair.to_csv(DATA/'R56_05_逐年份种子提前期成对改善.csv',index=False,encoding='utf-8-sig')
    checks=[]
    for model in QA_MODELS:
        sub=pair[pair['模型']==model]
        byyear=sub.groupby('QA年份')[['DP相对Original_公共冻结改善_km','DP相对Original_原始流轴改善_km']].mean()
        bylead=sub.groupby('提前期_天')[['DP相对Original_公共冻结改善_km','DP相对Original_原始流轴改善_km']].mean()
        mean_f=float(sub['DP相对Original_公共冻结改善_km'].mean())
        mean_r=float(sub['DP相对Original_原始流轴改善_km'].mean())
        all_years=bool((byyear['DP相对Original_公共冻结改善_km']>0).all())
        all_leads=bool((bylead['DP相对Original_公共冻结改善_km']>=0).all())
        raw_gain=bool(np.isfinite(mean_r) and mean_r>0)
        checks.append({'模型':model,'冻结目标平均改善_km':mean_f,'原始流轴抽样平均改善_km':mean_r,
            'QA两年均改善':all_years,'三个提前期都不退化':all_leads,'抽样原始流轴改善':raw_gain,
            '是否有条件进入独立复制实验':bool(mean_f>=0.5 and all_years and all_leads and raw_gain)})
    gate=pd.DataFrame(checks).sort_values('冻结目标平均改善_km',ascending=False)
    gate.to_csv(DATA/'R56_05_表示机制资格门槛.csv',index=False,encoding='utf-8-sig')
    # 论文最终关心的强基线对照：在相同DP表示、相同QA年份和共同冻结参考下，混合结构是否优于LSTM
    dp_main=run[run['表示']=='DP-B2']
    hh=dp_main.pivot_table(index=['QA年份','种子','提前期_天'],columns='模型',values='公共冻结RMSE_km')
    hh['较强容量匹配LSTM']=hh[['Matched-LSTM','R53-CapacityMatchedLSTM']].min(axis=1)
    hh['混合结构相对LSTM改善_km']=hh['较强容量匹配LSTM']-hh['R53-ChannelConv-PCConcat']
    hh.reset_index()[['QA年份','种子','提前期_天','混合结构相对LSTM改善_km']].to_csv(
        DATA/'R56_06_DP表示混合结构相对LSTM成对改善.csv',index=False,encoding='utf-8-sig')
    improved=float(hh['混合结构相对LSTM改善_km'].mean())
    bylead_hh=hh.groupby('提前期_天')['混合结构相对LSTM改善_km'].mean()
    byyear_hh=hh.groupby('QA年份')['混合结构相对LSTM改善_km'].mean()
    hybrid_vs_lstm_gate=bool(improved>=0.5 and (bylead_hh>=0).all() and (byyear_hh>0).all())
    decision={'geometry_gate':geom_stats, 'development_QA_years':[2008,2016],
       'repeat_exposure_not_blind':True,'geometry_pilot_only':True,
       'qualified_candidates':gate.loc[gate['是否有条件进入独立复制实验'],'模型'].tolist(),
       'dp_hybrid_minus_lstm_common_frozen_improvement_km':improved,
       'dp_hybrid_vs_lstm_both_years_and_leads_gate':hybrid_vs_lstm_gate,
       'formal_training_qualified':False,
       'note':'即使DP模型提升且混合结构胜过LSTM，也不直接启动正式多年后报；需额外确认与未接触年份的公开报告。'}
    (REP/'R56_最终交接.json').write_text(json.dumps(decision,indent=2,ensure_ascii=False),encoding='utf-8')
    display(mean);display(gate)
    # 双语图；只显示公共冻结参考，不拿各自目标误差混合排序
    palette={'Matched-LSTM':'#7fa8bc','R53-CapacityMatchedLSTM':'#9ab7c5','EOF-TCN-LSTM':'#537b96','R53-ChannelConv-PCConcat':'#d49a6a'}
    for zh,dir in ((True,FZH),(False,FEN)):
        old=plt.rcParams.copy()
        plt.rcParams['font.family']=[CJK_NAME] if zh else ['DejaVu Sans']
        plt.rcParams['axes.unicode_minus']=False
        fig,ax=plt.subplots(figsize=(8.0,4.5))
        for model in QA_MODELS:
            for rep in QA_REPS:
                sub=mean[(mean['模型']==model)&(mean['表示']==rep)].sort_values('提前期_天')
                ax.plot(sub['提前期_天'],sub['公共冻结RMSE_km'],marker='o',linestyle='-' if rep=='Original' else '--',
                  color=palette[model],lw=1.7,label=model+' / '+rep)
        ax.set_xlabel('提前期（天）' if zh else 'Forecast lead (days)')
        ax.set_ylabel('冻结参考路径 RMSE (km)' if zh else 'Frozen-reference path RMSE (km)')
        ax.set_title('R5.6 相同参考路径下的表示消融' if zh else 'R5.6 representation ablation on a common reference')
        ax.grid(alpha=.2);ax.legend(fontsize=7,ncol=2);fig.tight_layout()
        suff='中文' if zh else 'English'
        fig.savefig(dir/f'图1_R56_共同参考路径误差_{suff}.png',dpi=cfg.FIG_DPI,bbox_inches='tight')
        fig.savefig(dir/f'图1_R56_共同参考路径误差_{suff}.pdf',bbox_inches='tight')
        plt.close(fig);plt.rcParams.update(old)
    r56_archive('completed_and_evaluated')

## 研究合同与限制

1. **不得覆盖** `final/dataset/...FINAL.npz` 与 `final/tracks_final`。DP 候选只在 `R5_6_.../派生表示_不覆盖冻结数据/`。
2. **训练统一时序合同**：180 天历史，+30/+60/+90 天；EOF、标准化只拟合每折训练期；内层早停使用上一年，QA 仅作最后评价；`RUN_FULL_TRAINING=False`。
3. **三个评价口径必须同时读**：统一的旧冻结目标（模型对照）、各自表示目标（诊断）、抽样原始流轴几何距离（内部物理参考）。不可把后两者误称独立观测真值。
4. **R5.6 结论有限**：所有开发 QA 年份都已经被研究人员看过，即使本轮增益很好，也不能称盲测，只能支持后续独立复制。
5. 生成文件 `D:\KuroshioDaily1993_2025\R56_上传给ChatGPT_几何表示机制实验结果.zip`，Notebook 提供可点击本机链接，若 VS Code 链接不响应，使用文件资源管理器打开。